# CRAC-01 failure prediction — model training

Trains the primary 4-hour failure classifier and the time-to-failure
regressor described in plan section 5. Two methodological rules from
section 5.2 are load-bearing and must not be skipped:

1. **Split on `run_id`, never on rows.** Consecutive 30-second samples
   are highly autocorrelated; a random row split leaks near-identical
   rows across train/test and the reported score collapses in real use.
2. **Drop underscore-prefixed columns** (`_bearing_wear`, `_filter_load`)
   before training — they are simulator ground truth, not observable
   telemetry, and using them is leakage.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score, mean_absolute_error
)
import joblib

RANDOM_SEED = 42  # dataset is reproducible from this seed alone — CSV need not be committed

## 1. Generate the dataset

`sensor_simulator.py` is a live, real-time, single-lifecycle demo
tool — it has no run structure or failure labels, so it can't
directly produce a training set. `dataset/generate_dataset.py` runs
many simulated lifecycles in batch instead, labeled against each
lifecycle's actual outcome. Reproducible from `seed` alone (section
5.2) — nothing here needs to be committed as a CSV.

In [2]:
import sys
sys.path.insert(0, "..")  # repo root, so `dataset` package resolves
from dataset.generate_dataset import simulate_runs

df = simulate_runs(seed=RANDOM_SEED, n_runs=140)
print(df.shape)
print(df["_fault_type"].value_counts())
df.head()

(161546, 21)
_fault_type
bearing    61864
healthy    57365
filter     42317
Name: count, dtype: int64


,run_id,t,fan_rpm,fan_motor_current_a,fan_motor_temp_c,fan_vibration_mm_s,filter_dp_pa,airflow_cfm,supply_air_temp_c,return_air_temp_c,...,load_factor,_bearing_wear,_filter_load,_fault_type,time_to_failure_min,failure_within_30,failure_within_60,failure_within_120,failure_within_240,predicted_temp_60s
0,0,0,3165.0,4.49,66.5,1.84,115.5,3404.0,17.8,27.2,...,0.985,0.0123,0.0000,filter,265.92626,0,0,0,0,66.3
1,0,30,3134.0,4.53,66.4,2.01,120.3,3380.0,17.9,26.7,...,0.985,0.0409,0.0000,filter,265.42626,0,0,0,0,69.3
2,0,60,3174.0,4.56,66.3,1.90,117.7,3358.0,18.2,27.0,...,0.986,0.0312,0.0000,filter,264.92626,0,0,0,0,68.0
3,0,90,3128.0,4.57,69.3,2.02,123.9,3367.0,18.1,26.8,...,0.986,0.0724,0.0096,filter,264.42626,0,0,0,0,70.0
4,0,120,3167.0,4.46,68.0,1.92,117.6,3407.0,18.1,27.1,...,0.986,0.0483,0.0000,filter,263.92626,0,0,0,0,65.5


## 2. Feature engineering

Rolling-window slope of motor temperature and filter differential
pressure over the preceding ten minutes (20 samples at 30s interval).
Project 1 already predicted from a rolling slope: the Segment 3
dashboard tracked inlet-temperature slope and raised its predictive
alert at 0.2 °C/min toward the 30 °C inlet limit
(`Segment 3 - Dashboard/dashboard.html:1597`). The same technique is
generalised here to the CRAC's motor temperature and filter
differential pressure. See "Continuity with Project 1" in the report.

In [3]:
ROLL_WINDOW = 20  # 10 minutes at 30s sample interval

def add_rolling_features(df):
    df = df.sort_values(["run_id", "t"])
    df["motor_temp_slope_10min"] = (
        df.groupby("run_id")["fan_motor_temp_c"]
        .transform(lambda s: s.diff(ROLL_WINDOW) / ROLL_WINDOW)
    )
    df["filter_dp_slope_10min"] = (
        df.groupby("run_id")["filter_dp_pa"]
        .transform(lambda s: s.diff(ROLL_WINDOW) / ROLL_WINDOW)
    )
    return df

df = add_rolling_features(df)

## 3. Drop leakage columns, define feature set

In [4]:
# _fault_type is ground truth about run OUTCOME (not a per-sample
# observable), so it's excluded from training features below — but
# unlike _bearing_wear/_filter_load it's still useful at evaluation
# time to identify healthy runs for the false-alarm-rate check, so it
# is NOT dropped from the dataframe here, only kept out of FEATURE_COLS.
LEAKAGE_COLS = [c for c in df.columns if c.startswith("_") and c != "_fault_type"]
print("Dropping (ground truth, not observable):", LEAKAGE_COLS)

FEATURE_COLS = [
    "fan_rpm", "fan_motor_current_a", "fan_motor_temp_c", "fan_vibration_mm_s",
    "filter_dp_pa", "airflow_cfm", "supply_air_temp_c", "return_air_temp_c",
    "compressor_load_pct", "motor_temp_slope_10min", "filter_dp_slope_10min",
]

TARGET_CLS = "failure_within_240"   # primary: 4-hour horizon
TARGET_REG = "time_to_failure_min"

df = df.drop(columns=LEAKAGE_COLS)
# NOTE: dropna only on FEATURE_COLS + TARGET_CLS here. time_to_failure_min
# (TARGET_REG) is legitimately NaN for every row of a healthy run — it
# never fails, so there's no time-to-failure to report. Dropping on it
# here would silently delete all 39 healthy runs before the split even
# happens, which breaks both the split's stratification and the
# false-alarm-rate check further down.
df = df.dropna(subset=FEATURE_COLS + [TARGET_CLS])

Dropping (ground truth, not observable): ['_bearing_wear', '_filter_load']


## 4. Group split on run_id (70/30), stratified by outcome — not a row split

In [5]:
from sklearn.model_selection import ShuffleSplit
splitter = ShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_SEED)

# Stratify by _fault_type: split each outcome's run_ids separately, then
# combine. A plain group split over all 140 runs can, by chance, put
# zero of the 39 healthy runs in the 30% test fold — which silently
# breaks the false-alarm-rate check below. Splitting per stratum first
# guarantees every outcome is represented in both folds.
train_run_ids, test_run_ids = [], []
for fault_type, group_df in df.groupby("_fault_type"):
    run_ids = group_df["run_id"].unique()
    tr_idx, te_idx = next(splitter.split(run_ids))
    train_run_ids.extend(run_ids[tr_idx])
    test_run_ids.extend(run_ids[te_idx])

train_df = df[df["run_id"].isin(train_run_ids)]
test_df = df[df["run_id"].isin(test_run_ids)]

print(f"Train runs: {train_df['run_id'].nunique()}, Test runs: {test_df['run_id'].nunique()}")
# Printed above: 97 train / 43 test runs, every fault type in both folds

X_train, y_train_cls = train_df[FEATURE_COLS], train_df[TARGET_CLS]
X_test, y_test_cls = test_df[FEATURE_COLS], test_df[TARGET_CLS]

# Regression target only exists for failing runs (healthy runs have no
# time-to-failure) — filter to those rows specifically for the regressor.
train_reg_df = train_df.dropna(subset=[TARGET_REG])
test_reg_df = test_df.dropna(subset=[TARGET_REG])
X_train_reg, y_train_reg = train_reg_df[FEATURE_COLS], train_reg_df[TARGET_REG]
X_test_reg, y_test_reg = test_reg_df[FEATURE_COLS], test_reg_df[TARGET_REG]

Train runs: 97, Test runs: 43


## 5. Train the primary classifier — failure_within_240 (4-hour horizon)

The comparator for this project is not another model's score but the
threshold rules the system exists to beat; cell 19 measures both on the
same held-out runs. This classifier's metrics print below and are
carried into Table 1 of the report.

In [6]:
# in the notebook, replace the classifier cell:
clf = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight="balanced", random_state=42, n_jobs=-1)
clf.fit(X_train, y_train_cls)

proba = clf.predict_proba(X_test)[:, 1]
preds = (proba >= 0.5).astype(int)

print("AUC:      ", round(roc_auc_score(y_test_cls, proba), 3))
print("Precision:", round(precision_score(y_test_cls, preds), 3))
print("Recall:   ", round(recall_score(y_test_cls, preds), 3))
print("F1:       ", round(f1_score(y_test_cls, preds), 3))

AUC:       0.993
Precision: 0.892
Recall:    0.962
F1:        0.925


**False alarm rate** — alerts per healthy run, drives the
operator-trust argument per section 5.4.

In [7]:
# A healthy run is one whose _fault_type is "healthy" — i.e. it never
# actually fails, so ANY alert raised during it is a false alarm.
# (Using _fault_type here, at evaluation time, is fine — it's ground
# truth about run OUTCOME, not a feature the model saw during training.)
healthy_test_runs = test_df[test_df["_fault_type"] == "healthy"].copy()

if healthy_test_runs.empty:
    print("No healthy runs landed in the test split this time (small n_runs "
          "makes this possible with a group split) — re-run with a "
          "different random_state or a larger n_runs if you need this metric.")
else:
    healthy_test_runs["alert"] = clf.predict_proba(healthy_test_runs[FEATURE_COLS])[:, 1] >= 0.5
    false_alarms_per_run = healthy_test_runs.groupby("run_id")["alert"].sum()
    print("Mean false alarms per healthy run:", round(false_alarms_per_run.mean(), 2))
    print("Runs with at least one false alarm:",
          f"{(false_alarms_per_run > 0).sum()} / {len(false_alarms_per_run)}")

Mean false alarms per healthy run: 0.08
Runs with at least one false alarm: 1 / 12


## 6. Train the time-to-failure regressor

Baseline to beat: MAE 25.8 minutes (240-minute window).

Two changes from a naive first pass:
1. **Scope to the 240-minute window.** The baseline's own MAE is
   scoped to rows already within 240 minutes of failure — not the
   full multi-hour run. Training on the full run (where early rows
   have TTF in the hundreds of minutes) inflates the error on a
   problem the model was never asked to solve at that range.
2. **Gradient boosting instead of linear regression.** The wear
   curve is non-linear near failure — a linear model underfits that
   curvature specifically in the window that matters.

In [8]:
# Restrict to the scored window on both sides of the split.
WINDOW_MIN = 240
train_reg_df = train_reg_df[train_reg_df[TARGET_REG] <= WINDOW_MIN]
test_reg_df = test_reg_df[test_reg_df[TARGET_REG] <= WINDOW_MIN]
X_train_reg, y_train_reg = train_reg_df[FEATURE_COLS], train_reg_df[TARGET_REG]
X_test_reg, y_test_reg = test_reg_df[FEATURE_COLS], test_reg_df[TARGET_REG]

reg = GradientBoostingRegressor(
    n_estimators=300, max_depth=3, learning_rate=0.05, random_state=RANDOM_SEED
)
reg.fit(X_train_reg, y_train_reg)
reg_preds = reg.predict(X_test_reg)

mae = mean_absolute_error(y_test_reg, reg_preds)
print(f"MAE (minutes), {WINDOW_MIN}-min window, gradient boosting:", round(mae, 2))

MAE (minutes), 240-min window, gradient boosting: 17.26


### For comparison — same window, linear regression

Isolates how much of the improvement is the windowing vs. the model
swap.

In [9]:
from sklearn.linear_model import LinearRegression
linreg = LinearRegression()
linreg.fit(X_train_reg, y_train_reg)
linreg_mae = mean_absolute_error(y_test_reg, linreg.predict(X_test_reg))
print(f"MAE (minutes), {WINDOW_MIN}-min window, linear regression:", round(linreg_mae, 2))

MAE (minutes), 240-min window, linear regression: 31.16


## 7. Lead-time comparison — the headline number

Compares this model's usable lead time against the static health-state
threshold and the 60-second symptom-forecast baseline. This is the single
number that justifies the project.

The threshold baseline is **measured from the test set below**, not assumed.
An earlier version of this notebook divided by a hardcoded "~50 minutes"
carried over from the plan; the measured value is 195.2 minutes, which makes
the honest multiple 1.5x rather than the much larger number that assumption
implied. The cell below computes both medians on the same failing runs, so
the comparison is like for like.


In [10]:
def median_lead_time_minutes(df, alert_mask):
    """Median, across FAILING runs only, of how long before the actual
    failure a detector's first alert fired. A healthy run has no failure
    to measure lead time against, so it's excluded here.

    Also returns coverage: runs the detector never caught are excluded
    from the median rather than scored as zero, which flatters BOTH
    detectors, so the two coverage numbers must be read alongside the
    two medians.
    """
    failing = df[df["time_to_failure_min"].notna()].copy()
    failing["alert"] = alert_mask[failing.index]

    lead_times = []
    n_runs = failing["run_id"].nunique()
    for run_id, run_df in failing.groupby("run_id"):
        alerted = run_df[run_df["alert"]]
        if alerted.empty:
            continue  # never caught — excluded from the median
        # first alert = the row with the LARGEST remaining time-to-failure
        # among alerted rows (earliest in the run, most lead time)
        lead_times.append(alerted["time_to_failure_min"].max())

    if not lead_times:
        return None, 0, n_runs
    return round(float(np.median(lead_times)), 1), len(lead_times), n_runs


# --- Model detector -------------------------------------------------------
model_alert = pd.Series(
    clf.predict_proba(test_df[FEATURE_COLS])[:, 1] >= 0.5, index=test_df.index
)
model_lead_time, model_caught, n_failing = median_lead_time_minutes(test_df, model_alert)

# --- Static-threshold baseline, COMPUTED FROM THE DATA ---------------------
# Previously this cell divided by a hardcoded "~50 minutes" carried over
# from the plan document; it was never measured, so the headline multiple
# was not a measured comparison. These are the same plan section 4.3
# thresholds the CoolingTwin trips on, applied to the same test rows the
# model is scored on, through the same lead-time function.
MOTOR_TEMP_ALERT = 105.0
FILTER_DP_ALERT = 350.0
NOMINAL_AIRFLOW_CFM = 3400.0
AIRFLOW_TRIP_PCT = 0.65

threshold_alert = (
    (test_df["fan_motor_temp_c"] >= MOTOR_TEMP_ALERT)
    | (test_df["filter_dp_pa"] >= FILTER_DP_ALERT)
    | (test_df["airflow_cfm"] <= NOMINAL_AIRFLOW_CFM * AIRFLOW_TRIP_PCT)
)
baseline_lead_time, baseline_caught, _ = median_lead_time_minutes(test_df, threshold_alert)

print(f"Failing test runs: {n_failing}")
print(f"Model     median lead time: {model_lead_time} min "
      f"(caught {model_caught}/{n_failing} runs)")
print(f"Threshold median lead time: {baseline_lead_time} min "
      f"(caught {baseline_caught}/{n_failing} runs)")
if model_lead_time and baseline_lead_time:
    print(f"Lead-time multiple vs. MEASURED threshold baseline: "
          f"{round(model_lead_time / baseline_lead_time, 2)}x")


Failing test runs: 31
Model     median lead time: 292.5 min (caught 31/31 runs)
Threshold median lead time: 195.2 min (caught 31/31 runs)
Lead-time multiple vs. MEASURED threshold baseline: 1.5x


## 8. Save model artifacts

Committed alongside a sample of training data so results are
inspectable without retraining (section 12.2).

In [11]:
import os
os.makedirs("models", exist_ok=True)
joblib.dump(
    {"classifier": clf, "regressor": reg, "feature_cols": FEATURE_COLS},
    "models/crac_failure_model.joblib",
)
print("Saved to models/crac_failure_model.joblib")

Saved to models/crac_failure_model.joblib
